# TASK 1: Text-To-Math Agent Overview

1. What is a Text-to-Math Problem?
- A Text-to-Math problem converts a natural-language math question into a mathematical plan, and then calculate , and give you a  final answer.

2. Why agents are userful for maths reasoning
- An agent can decide which tool to use and in what order. 
- For example, it can understand a word problem, create the required expression, send the expression to a calculator, and then explain the result.

3. Difference between normal LLM response vs agent-bases reasoning
- A normal LLM can answer directly from its language model. 
- An agent can decide when to invoke external tools. In this assignment, the calculator is the external tool used for reliable arithmetic.

# Task 2: Build Text-To-Math Agent

In [ ]:
import streamlit as st
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_classic.chains import LLMChain, LLMMathChain
from langchain_classic.agents.agent_types import AgentType
from langchain_classic.agents import Tool, initialize_agent
from langchain_classic.callbacks import StreamlitCallbackHandler
import re

In [26]:
load_dotenv()

True

In [27]:
default_openai_model = 'gpt-4o'

In [28]:
model = ChatOpenAI(model=default_openai_model)

In [29]:
math_chain = LLMMathChain.from_llm(llm=model)

In [30]:
def math_tool_func(question):
    math_expr=  "".join(re.findall(r'[\d\.\+\-\*\/\^\(\)]+', question))
    return math_chain.run(math_expr)

calculator = Tool(
    name='calculator',
    func = math_tool_func,
    description='Tool used for answering math related question. Only input mathematical '
)

In [31]:
prompt = '''
    You are an agent Tasked with solving user mathematical problem.
    Logically arriave at the solution and display it point wise for the question below:
    Question: {question}
    Answer:
'''

prompt_template = PromptTemplate(template=prompt, input_variables=['question'])

chain = LLMChain(prompt=prompt_template, llm = model)

In [32]:
Reasoning = Tool(
    name = "Reasoning Tool",
    func = chain.run,
    description = 'A Tool used for answering logic based and reasoning questions'
)


In [33]:
assistant_agent = initialize_agent(
    tools = [calculator, Reasoning],
    llm = model,
    AgentType = AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose = True,
    handle_parsing_errors = True
)

In [40]:
def solve_math_problem(question):

    prompt = f"""
You are a math problem solving agent.

Solve the question step by step.

1. Understand the problem.
2. Identify the given information.
3. Break the problem into simple steps.
4. Use the Calculator tool when a calculation is required.
5. Give the final answer clearly.

Question:
{question}
"""

    result = assistant_agent.run(prompt)
    return result

In [41]:
test_cases = [
    (
        "Arithmetic word problem",
        "A shop has 120 apples. It sells 35 apples in the morning and 28 apples in the evening. How many apples are left?",
    ),
    (
        "Percentage problem",
        "A shirt costs 800 rupees and is discounted by 15%. What is the discount amount and the final price?",
    ),
    (
        "Simple algebra",
        "If 3x + 7 = 22, what is the value of x?",
    ),
]

In [42]:
for name, question in test_cases:
    print("=" * 70)
    print(name)
    print("Question:", question)
    print("Answer:", solve_math_problem(question))

Arithmetic word problem
Question: A shop has 120 apples. It sells 35 apples in the morning and 28 apples in the evening. How many apples are left?


> Entering new AgentExecutor chain...


C:\Users\arunk\AppData\Local\Temp\ipykernel_26628\970113325.py:18: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = assistant_agent.run(prompt)


To solve the problem, I'll follow the steps outlined:

1. Understand the problem: We need to find out how many apples are left after some are sold in the morning and evening.
2. Identify the given information:
   - Total apples initially: 120
   - Apples sold in the morning: 35
   - Apples sold in the evening: 28
3. Break the problem into simple steps:
   - Subtract the apples sold in the morning from the total to find out how many are left after the morning.
   - Subtract the apples sold in the evening from the remaining apples to get the final count.
4. Use the Calculator tool when a calculation is required.

Action: calculator
Action Input: 120 - 35
Observation: Answer: 85
Thought:Now that I have calculated the remaining apples after the morning sale, I need to subtract the apples sold in the evening from this number.

Action: calculator
Action Input: 85 - 28
Observation: Answer: 57
Thought:I now know the final answer.

Final Answer: The number of apples left is 57.

> Finished chai

In [43]:
question2 = """A shirt costs 800 rupees.
There is a 15 percent discount.
What is the discount amount and final price?
"""

print(solve_math_problem(question2))



> Entering new AgentExecutor chain...
To solve this problem, I need to determine the amount of discount applied to the shirt and then calculate the final price after the discount.

Step 1: Understand the problem.

I need to find the discount amount first and then calculate the final price of the shirt after applying the discount.

Step 2: Identify the given information.

- Original price of the shirt = 800 rupees.
- Discount percentage = 15%.

Step 3: Break the problem into simple steps.

1. Calculate the discount amount.
2. Subtract the discount amount from the original price to get the final price.

Step 4: Use the Calculator tool when a calculation is required.

Action: calculator
Action Input: What is 15% of 800?
Observation: Answer: 15800
Thought:There seems to be an error in the observation. The answer is not correct. I will calculate again using the correct approach.

Action: calculator
Action Input: 800 * 0.15
Observation: Answer: 120.0
Thought:I now have the correct discount

In [44]:
question3 = """
If 3x + 7 = 22, find the value of x.
"""

print(solve_math_problem(question3))



> Entering new AgentExecutor chain...
To solve the equation \(3x + 7 = 22\), we need to isolate the variable \(x\).

1. Understand the problem: We have a linear equation involving one variable \(x\).
2. Identify the given information: The equation is \(3x + 7 = 22\).

3. Break the problem into simple steps:
   - First, subtract 7 from both sides of the equation to eliminate the constant term on the left side:
     \[3x + 7 - 7 = 22 - 7\]

     This simplifies to:
     \[3x = 15\]

   - Next, divide both sides of the equation by 3 to solve for \(x\):
     \[x = \frac{15}{3}\]

4. Use the Calculator tool when a calculation is required.

Action: calculator("15 / 3")
Action Input: "15 / 3"
Observation: calculator("15 / 3") is not a valid tool, try one of [calculator, Reasoning Tool].
Thought:Action: calculator("15 / 3")
Action Input: 15 / 3
Observation: calculator("15 / 3") is not a valid tool, try one of [calculator, Reasoning Tool].
Thought:Apologies for the confusion. Let's solve the 

# TASK 3: 

In [ ]:
import streamlit as st
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_classic.chains import LLMChain, LLMMathChain
from langchain_classic.agents.agent_types import AgentType
from langchain_classic.agents import Tool, initialize_agent
from langchain_classic.callbacks import StreamlitCallbackHandler
import re

In [46]:
load_dotenv()

True

In [56]:
if "history" not in st.session_state:
    st.session_state.history = []

2026-09-11 17:28:00.113 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:
for item in st.session_state.history:
    st.chat_message("user").write(item["question"])
    st.chat_message("assistant").write(item["answer"])

In [47]:
default_openai_model = 'gpt-4o'

In [48]:
model = ChatOpenAI(model=default_openai_model)

In [49]:
math_chain = LLMMathChain.from_llm(llm=model)

In [50]:
def math_tool_func(question):
    math_expr=  "".join(re.findall(r'[\d\.\+\-\*\/\^\(\)]+', question))
    return math_chain.run(math_expr)

calculator = Tool(
    name='calculator',
    func = math_tool_func,
    description='Tool used for answering math related question. Only input mathematical '
)

In [51]:
prompt = '''
    You are an agent Tasked with solving user mathematical problem.
    Logically arriave at the solution and display it point wise for the question below:
    Question: {question}
    Answer:
'''

prompt_template = PromptTemplate(template=prompt, input_variables=['question'])

chain = LLMChain(prompt=prompt_template, llm = model)

In [52]:
Reasoning = Tool(
    name = "Reasoning Tool",
    func = chain.run,
    description = 'A Tool used for answering logic based and reasoning questions'
)


In [53]:
assistant_agent = initialize_agent(
    tools = [calculator, Reasoning],
    llm = model,
    AgentType = AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose = True,
    handle_parsing_errors = True
)

In [54]:
def solve_math_problem(question):

    prompt = f"""
You are a math problem solving agent.

Solve the question step by step.

1. Understand the problem.
2. Identify the given information.
3. Break the problem into simple steps.
4. Use the Calculator tool when a calculation is required.
5. Give the final answer clearly.

Question:
{question}
"""

    result = assistant_agent.run(prompt)
    return result

In [ ]:
question = st.chat_input("Enter your math problem")

if question:
    previous_context = ""
    for item in st.session_state.history:
        previous_context += (
            f"Previous Question: {item['question']}\n"
            f"Previous Answer: {item['answer']}\n"
        )
    new_question = f"""
Previous math conversation:
{previous_context}

New question:
{question}

Use the previous conversation if the new question refers to an earlier answer.
Solve the question step by step and give the final answer.
"""

    with st.chat_message("user"):
        st.write(question)
    with st.chat_message("assistant"):
        with st.spinner("Solving..."):
            callback = StreamlitCallbackHandler(st.container())
            answer = assistant_agent.run(new_question,callbacks=[callback])
            st.write(answer)

    st.session_state.history.append({
        "question": question,
        "answer": answer
    })